# Global INT8 Quantization

Bot task id: `10_quantize_global_int8`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/10_quantize_global_int8.ipynb)

## Условие

Глобальная (per-tensor) симметричная квантизация тензора в int8. Дан тензор `X` (fp16/fp32). Вернуть квантизованный `Q` (int8) и `absmax` (скаляр той же точности, что `X`).

$$\text{absmax} = \max |X|, \qquad \text{absmax\_inv} = 1 / \text{absmax}, \qquad Q = \operatorname{round}(127 \cdot X \cdot \text{absmax\_inv})$$

**Требования**
- одно Triton-ядро, линейный проход по массиву с маскированием хвоста;
- `tl.load` / `tl.store`;
- `@triton.autotune`, как минимум 2 конфига;
- обёртка `quantize_global(x)` возвращает `(q_int8, absmax)`;
- вход: CUDA, `torch.float16` или `torch.float32`; выход: `q.dtype == torch.int8`, `absmax.shape == (1,)`, `absmax.dtype == x.dtype`.

`absmax` в сигнатуру ядра не передаётся, только `absmax_inv_ptr`, так что максимум можно посчитать в обёртке.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `10_quantize_global_int8.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 10_quantize_global_int8.py
import torch
import triton
import triton.language as tl


@triton.autotune(
    configs=[
        triton.Config({"BLOCK_SIZE": 1024}, num_warps=4),
        triton.Config({"BLOCK_SIZE": 2048}, num_stages=1),
    ],
    key=["n_elements"],
)
@triton.jit
def _quantize_global(x_ptr, absmax_inv_ptr, output_ptr, n_elements, BLOCK_SIZE: tl.constexpr):
    # YOUR CODE HERE
    pass


def quantize_global(x: torch.Tensor):
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "10_quantize_global_int8.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

def quant_ref(x, denom):
    return torch.round(127.0 * x.float() / denom).clamp(-127, 127).to(torch.int8)


def assert_quant(q, ref):
    # int8 values may differ by 1 where rounding modes disagree on .5
    assert q.dtype == torch.int8, q.dtype
    assert q.shape == ref.shape, (q.shape, ref.shape)
    diff = (q.int() - ref.int()).abs()
    assert diff.max() <= 1, f"max |q - ref| = {diff.max().item()}"
    assert (diff > 0).float().mean() < 1e-3, f"{(diff > 0).sum().item()} elements off by 1"

for dtype in (torch.float32, torch.float16):
    x = torch.randn(3, 1000, 7, device="cuda", dtype=dtype)
    q, absmax = sol.quantize_global(x)
    ref_max = x.abs().max()
    assert absmax.shape == (1,) and absmax.dtype == dtype, (absmax.shape, absmax.dtype)
    torch.testing.assert_close(absmax[0], ref_max)
    assert_quant(q, quant_ref(x, ref_max.float()))
assert len(sol._quantize_global.configs) >= 2, "the task asks for at least 2 autotune configs"

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("10_quantize_global_int8.py")